In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import ast
from scipy import stats
import librosa
import warnings
print(os.getcwd())

/home/jakub-karczewski/Dokumenty/data_exploration_project


In [2]:
paths = {}
for root, dirs, files in os.walk(os.path.join(os.getcwd(), "fma_small")):
    if root.endswith("fma_small"):
        continue
    for file in files:
        full_path = os.path.join(root, file)
        track_id = int(file.split(".")[0])
        paths[track_id] = full_path

In [3]:
paths
len(list(paths.keys()))

8000

In [4]:
def columns():
    feature_sizes = dict(
        chroma_stft=12,
        mfcc=20, rms=1, zcr=1,
        spectral_centroid=1
    )
    moments = ('mean', 'std')

    tuples = [
        (feature, moment, f"{i+1:02d}")
        for feature, size in feature_sizes.items()
        for moment in moments
        for i in range(size)
    ]

    return pd.MultiIndex.from_tuples(tuples, names=('feature', 'statistics', 'number')).sort_values()

In [6]:
!pip install essentia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 6.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 7.6 MB/s eta 0:00:0000:0100:01


In [5]:
import essentia.standard as es
def compute_features_essentia(tid):
    features = pd.Series(index=columns(), dtype=np.float32, name=tid)
    
    def feature_stats(name, values):
        values = np.array(values)
        if values.ndim == 1:
            values = values.reshape(-1, 1)
        for i in range(values.shape[0]):
            features[name, 'mean', f'{i+1:02d}'] = np.float32(np.mean(values[i, :]))
            features[name, 'std', f'{i+1:02d}'] = np.float32(np.std(values[i, :]))
            
    try:
        filepath = paths[tid]

        loader = es.MonoLoader(filename=filepath, sampleRate=22050)
        x = loader()

        frame_size = 2048
        hop_size = 512

        window = es.Windowing(type='hann')
        spectrum = es.Spectrum()

        zcr_vals = []
        rms_vals = []
        centroid_vals = []
        chroma_vals = []
        mfcc_vals = []

        mfcc = es.MFCC(numberCoefficients=20)
        spectral_peaks = es.SpectralPeaks()
        hpcp = es.HPCP(size=12)

        for frame in es.FrameGenerator(x, frameSize=frame_size, hopSize=hop_size):

            zcr_vals.append(es.ZeroCrossingRate()(frame))

            w = window(frame)
            spec = spectrum(w)

            rms_vals.append(es.RMS()(frame))

            centroid_vals.append(es.Centroid()(spec))

            freqs, mags = spectral_peaks(spec)
            chroma_vals.append(hpcp(freqs, mags))

            # MFCC (includes mel bands internally)
            mfcc_bands, mfcc_coeffs = mfcc(spec)
            mfcc_vals.append(mfcc_coeffs)

        feature_stats('zcr', np.array(zcr_vals))
        feature_stats('rms', np.array(rms_vals))
        feature_stats('spectral_centroid', np.array(centroid_vals))
        feature_stats('chroma_stft', np.array(chroma_vals))
        feature_stats('mfcc', np.array(mfcc_vals))

    except Exception as e:
        print(f'{tid}: {repr(e)}')

    return features
    
    

[   INFO   ] MusicExtractorSVM: no classifier models were configured by default


### To be used on Linux

In [ ]:
from multiprocessing import Pool
from tqdm import tqdm
import os

track_ids = list(paths.keys())

features = pd.DataFrame(
    index=track_ids,
    columns=columns(),
    dtype=np.float32
)

nb_workers = os.cpu_count() 

with Pool(nb_workers) as pool:
    for row in tqdm(pool.imap_unordered(compute_features_essentia, track_ids),
                    total=len(track_ids)):
        features.loc[row.name] = row


features.to_csv(
    "features_small_essentia.csv",
    float_format="%.6f",
    header=['_'.join(col).strip() for col in features.columns.values],
    index_label="track_id"
)

 35%|███▍      | 2785/8000 [33:43<45:40,  1.90it/s]  

108925: RuntimeError('Error while configuring MonoLoader: AudioLoader: Could not find stream information, error = End of file')


 88%|████████▊ | 7023/8000 [1:23:46<05:20,  3.05it/s]

99134: RuntimeError('Error while configuring MonoLoader: AudioLoader: Could not find stream information, error = End of file')


 93%|█████████▎| 7477/8000 [1:29:06<04:14,  2.06it/s]

133297: RuntimeError('Error while configuring MonoLoader: AudioLoader: Could not find stream information, error = End of file')


100%|██████████| 8000/8000 [1:35:07<00:00,  1.40it/s]


In [ ]:
features.head()